In [5]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

In [6]:
import pandas as pd
from src.config import CLEAN_COMMENTS_PATH, COMMENTS_JSON_PATH

In [4]:
import sys
print(sys.path)

['c:\\Users\\thioy\\AppData\\Local\\Programs\\Python\\Python311\\python311.zip', 'c:\\Users\\thioy\\AppData\\Local\\Programs\\Python\\Python311\\Lib', 'c:\\Users\\thioy\\AppData\\Local\\Programs\\Python\\Python311\\DLLs', '', 'C:\\Users\\thioy\\AppData\\Roaming\\Python\\Python311\\site-packages', 'C:\\Users\\thioy\\AppData\\Roaming\\Python\\Python311\\site-packages\\win32', 'C:\\Users\\thioy\\AppData\\Roaming\\Python\\Python311\\site-packages\\win32\\lib', 'C:\\Users\\thioy\\AppData\\Roaming\\Python\\Python311\\site-packages\\Pythonwin', 'c:\\Users\\thioy\\AppData\\Local\\Programs\\Python\\Python311', 'c:\\Users\\thioy\\AppData\\Local\\Programs\\Python\\Python311\\Lib\\site-packages', 'c:\\Users\\thioy\\AppData\\Local\\Programs\\Python\\Python311\\Lib\\site-packages\\IPython\\extensions', 'C:\\Users\\thioy\\.ipython']


In [2]:
data = pd.read_json(COMMENTS_JSON_PATH, lines=True)
data.head()

,text,video_url
0,❤❤❤❤❤,https://www.youtube.com/watch?v=Myd-Gg4Mhck
1,❤️‍🩹🥳🥳🥳🥳🥳🥳,https://www.youtube.com/watch?v=Myd-Gg4Mhck
2,5:10,https://www.youtube.com/watch?v=Myd-Gg4Mhck
3,4:17,https://www.youtube.com/watch?v=Myd-Gg4Mhck
4,Non mane j'aime khalile❤😂,https://www.youtube.com/watch?v=Myd-Gg4Mhck


In [3]:
data.shape

(396559, 2)

Remove emojis, one word sentences, @, french sentences, numbers or time stamps (10:17)

In [4]:
import re

def clean_comment(text):
    
    # Remove URLs (often contain mixed case, best to remove early)
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    
    # Usernames (handles with @, including hyphens)
    text = re.sub(r'@[\w-]+', '', text)
    
    # Timestamps (HH:MM or HH:MM:SS)
    text = re.sub(r'\b\d{1,2}:\d{2}(:\d{2})?\b', '', text)
    
    # Emojis (Unicode-safe, keeps accented characters like è)
    emoji_pattern = re.compile("[\U00010000-\U0010ffff\u2600-\u27bf]+", flags=re.UNICODE)
    text = emoji_pattern.sub('', text)
    
    # Lowercase the text
    text = text.lower()
    
    # Normalize Punctuation (reduce !!! to ! and ??? to ?)
    text = re.sub(r'(!){2,}', r'\1', text)
    text = re.sub(r'(\?){2,}', r'\1', text)
    
    # Final Whitespace Cleanup
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text


In [5]:
data["clean_comment"] = data["text"].apply(clean_comment)

In [6]:
data.head(10)

,text,video_url,clean_comment
0,❤❤❤❤❤,https://www.youtube.com/watch?v=Myd-Gg4Mhck,
1,❤️‍🩹🥳🥳🥳🥳🥳🥳,https://www.youtube.com/watch?v=Myd-Gg4Mhck,️‍
2,5:10,https://www.youtube.com/watch?v=Myd-Gg4Mhck,
3,4:17,https://www.youtube.com/watch?v=Myd-Gg4Mhck,
4,Non mane j'aime khalile❤😂,https://www.youtube.com/watch?v=Myd-Gg4Mhck,non mane j'aime khalile
5,Manchalla ❤❤❤❤❤❤❤,https://www.youtube.com/watch?v=Myd-Gg4Mhck,manchalla
6,❤❤❤❤,https://www.youtube.com/watch?v=Myd-Gg4Mhck,
7,43:43 je n’ai jamais vu une famille aussi intr...,https://www.youtube.com/watch?v=Myd-Gg4Mhck,je n’ai jamais vu une famille aussi intrusive ...
8,36:59 Lekhou la beug man😂😂\nMachallah nala thi...,https://www.youtube.com/watch?v=Myd-Gg4Mhck,lekhou la beug man machallah nala thiate sekhe...
9,❤❤❤❤ tu es une fille ou,https://www.youtube.com/watch?v=Myd-Gg4Mhck,tu es une fille ou


In [7]:
clean_data = data.loc[~data["clean_comment"].isin(["", " "])]
clean_data

,text,video_url,clean_comment
1,❤️‍🩹🥳🥳🥳🥳🥳🥳,https://www.youtube.com/watch?v=Myd-Gg4Mhck,️‍
4,Non mane j'aime khalile❤😂,https://www.youtube.com/watch?v=Myd-Gg4Mhck,non mane j'aime khalile
5,Manchalla ❤❤❤❤❤❤❤,https://www.youtube.com/watch?v=Myd-Gg4Mhck,manchalla
7,43:43 je n’ai jamais vu une famille aussi intr...,https://www.youtube.com/watch?v=Myd-Gg4Mhck,je n’ai jamais vu une famille aussi intrusive ...
8,36:59 Lekhou la beug man😂😂\nMachallah nala thi...,https://www.youtube.com/watch?v=Myd-Gg4Mhck,lekhou la beug man machallah nala thiate sekhe...
...,...,...,...
396554,Toppou leine sama guinaw,https://www.youtube.com/watch?v=aj4wylbWqgs,toppou leine sama guinaw
396555,Bjr,https://www.youtube.com/watch?v=aj4wylbWqgs,bjr
396556,Fayou niou la,https://www.youtube.com/watch?v=aj4wylbWqgs,fayou niou la
396557,@Bambadieng-f2r for soula 😂😂,https://www.youtube.com/watch?v=aj4wylbWqgs,for soula


In [8]:
clean_data = clean_data.loc[clean_data["clean_comment"].str.split().str.len() > 2]
clean_data

,text,video_url,clean_comment
4,Non mane j'aime khalile❤😂,https://www.youtube.com/watch?v=Myd-Gg4Mhck,non mane j'aime khalile
7,43:43 je n’ai jamais vu une famille aussi intr...,https://www.youtube.com/watch?v=Myd-Gg4Mhck,je n’ai jamais vu une famille aussi intrusive ...
8,36:59 Lekhou la beug man😂😂\nMachallah nala thi...,https://www.youtube.com/watch?v=Myd-Gg4Mhck,lekhou la beug man machallah nala thiate sekhe...
9,❤❤❤❤ tu es une fille ou,https://www.youtube.com/watch?v=Myd-Gg4Mhck,tu es une fille ou
10,Wouyyy bb mareme😂😂😂,https://www.youtube.com/watch?v=Myd-Gg4Mhck,wouyyy bb mareme
...,...,...,...
396539,Cv nouyou na len ❤,https://www.youtube.com/watch?v=aj4wylbWqgs,cv nouyou na len
396540,Fekk lennma fii,https://www.youtube.com/watch?v=aj4wylbWqgs,fekk lennma fii
396547,Première commentaire ❤domou Guediawaye,https://www.youtube.com/watch?v=aj4wylbWqgs,première commentaire domou guediawaye
396554,Toppou leine sama guinaw,https://www.youtube.com/watch?v=aj4wylbWqgs,toppou leine sama guinaw


In [9]:
from langdetect import detect_langs

In [10]:
comment = "Mère oumy guène foke ni wérr na"
words = comment.split()
french_ratio = sum(1 for word in words if detect_langs(word)[0].lang == 'fr') / len(words)
print(f"French Ratio: {french_ratio:.2f}")


French Ratio: 0.29


In [11]:
def french_ratio(comment):
    try:
        words = comment.split()

        if len(words) == 0:
            return "to be removed"

        valid_words = []
        ignored_count = 0

        for word in words:
            # Keep only alphabetic characters (including accents)
            cleaned = re.sub(r"[^a-zA-ZÀ-ÿ]", "", word)

            # Ignore empty results or pure numbers/symbols
            if cleaned == "":
                ignored_count += 1
                continue

            valid_words.append(cleaned)

        # Remove if more than half is noise
        if ignored_count / len(words) > 0.5:
            return "to be removed"

        # Remove if nothing usable remains
        if len(valid_words) == 0:
            return "to be removed"

        fr_count = 0
        processed_count = 0

        for word in valid_words:
            try:
                detected = detect_langs(word)

                if detected and detected[0].lang == "fr":
                    fr_count += 1

                processed_count += 1

            except:
                continue

        # Remove if nothing could be processed
        if processed_count == 0:
            return "to be removed"

        return fr_count / processed_count

    except Exception as e:
        print(f"Error happened for comment: {comment}")
        print(e)
        return "to be removed"

In [13]:
from tqdm.auto import tqdm

tqdm.pandas()

c:\Users\thioy\Desktop\School\PFE\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [14]:
clean_data["french_ratio"] = clean_data["clean_comment"].progress_apply(french_ratio)
clean_data

100%|██████████| 257270/257270 [3:25:33<00:00, 20.86it/s]   


,text,video_url,clean_comment,french_ratio
4,Non mane j'aime khalile❤😂,https://www.youtube.com/watch?v=Myd-Gg4Mhck,non mane j'aime khalile,0.0
7,43:43 je n’ai jamais vu une famille aussi intr...,https://www.youtube.com/watch?v=Myd-Gg4Mhck,je n’ai jamais vu une famille aussi intrusive ...,0.37931
8,36:59 Lekhou la beug man😂😂\nMachallah nala thi...,https://www.youtube.com/watch?v=Myd-Gg4Mhck,lekhou la beug man machallah nala thiate sekhe...,0.0
9,❤❤❤❤ tu es une fille ou,https://www.youtube.com/watch?v=Myd-Gg4Mhck,tu es une fille ou,0.6
10,Wouyyy bb mareme😂😂😂,https://www.youtube.com/watch?v=Myd-Gg4Mhck,wouyyy bb mareme,0.0
...,...,...,...,...
396539,Cv nouyou na len ❤,https://www.youtube.com/watch?v=aj4wylbWqgs,cv nouyou na len,0.25
396540,Fekk lennma fii,https://www.youtube.com/watch?v=aj4wylbWqgs,fekk lennma fii,0.0
396547,Première commentaire ❤domou Guediawaye,https://www.youtube.com/watch?v=aj4wylbWqgs,première commentaire domou guediawaye,0.5
396554,Toppou leine sama guinaw,https://www.youtube.com/watch?v=aj4wylbWqgs,toppou leine sama guinaw,0.25


In [17]:
clean_data = clean_data.loc[clean_data["french_ratio"] != "to be removed"]
clean_data

,text,video_url,clean_comment,french_ratio
4,Non mane j'aime khalile❤😂,https://www.youtube.com/watch?v=Myd-Gg4Mhck,non mane j'aime khalile,0.0
7,43:43 je n’ai jamais vu une famille aussi intr...,https://www.youtube.com/watch?v=Myd-Gg4Mhck,je n’ai jamais vu une famille aussi intrusive ...,0.37931
8,36:59 Lekhou la beug man😂😂\nMachallah nala thi...,https://www.youtube.com/watch?v=Myd-Gg4Mhck,lekhou la beug man machallah nala thiate sekhe...,0.0
9,❤❤❤❤ tu es une fille ou,https://www.youtube.com/watch?v=Myd-Gg4Mhck,tu es une fille ou,0.6
10,Wouyyy bb mareme😂😂😂,https://www.youtube.com/watch?v=Myd-Gg4Mhck,wouyyy bb mareme,0.0
...,...,...,...,...
396539,Cv nouyou na len ❤,https://www.youtube.com/watch?v=aj4wylbWqgs,cv nouyou na len,0.25
396540,Fekk lennma fii,https://www.youtube.com/watch?v=aj4wylbWqgs,fekk lennma fii,0.0
396547,Première commentaire ❤domou Guediawaye,https://www.youtube.com/watch?v=aj4wylbWqgs,première commentaire domou guediawaye,0.5
396554,Toppou leine sama guinaw,https://www.youtube.com/watch?v=aj4wylbWqgs,toppou leine sama guinaw,0.25


In [ ]:
clean_data['french_ratio'] = pd.to_numeric(clean_data['french_ratio'], errors='coerce')

,text,video_url,clean_comment,french_ratio
4,Non mane j'aime khalile❤😂,https://www.youtube.com/watch?v=Myd-Gg4Mhck,non mane j'aime khalile,0.00000
7,43:43 je n’ai jamais vu une famille aussi intr...,https://www.youtube.com/watch?v=Myd-Gg4Mhck,je n’ai jamais vu une famille aussi intrusive ...,0.37931
8,36:59 Lekhou la beug man😂😂\nMachallah nala thi...,https://www.youtube.com/watch?v=Myd-Gg4Mhck,lekhou la beug man machallah nala thiate sekhe...,0.00000
9,❤❤❤❤ tu es une fille ou,https://www.youtube.com/watch?v=Myd-Gg4Mhck,tu es une fille ou,0.60000
10,Wouyyy bb mareme😂😂😂,https://www.youtube.com/watch?v=Myd-Gg4Mhck,wouyyy bb mareme,0.00000
...,...,...,...,...
396539,Cv nouyou na len ❤,https://www.youtube.com/watch?v=aj4wylbWqgs,cv nouyou na len,0.25000
396540,Fekk lennma fii,https://www.youtube.com/watch?v=aj4wylbWqgs,fekk lennma fii,0.00000
396547,Première commentaire ❤domou Guediawaye,https://www.youtube.com/watch?v=aj4wylbWqgs,première commentaire domou guediawaye,0.50000
396554,Toppou leine sama guinaw,https://www.youtube.com/watch?v=aj4wylbWqgs,toppou leine sama guinaw,0.25000


In [22]:
clean_data = clean_data.loc[clean_data["french_ratio"]< 0.5]
clean_data

,text,video_url,clean_comment,french_ratio
4,Non mane j'aime khalile❤😂,https://www.youtube.com/watch?v=Myd-Gg4Mhck,non mane j'aime khalile,0.00000
7,43:43 je n’ai jamais vu une famille aussi intr...,https://www.youtube.com/watch?v=Myd-Gg4Mhck,je n’ai jamais vu une famille aussi intrusive ...,0.37931
8,36:59 Lekhou la beug man😂😂\nMachallah nala thi...,https://www.youtube.com/watch?v=Myd-Gg4Mhck,lekhou la beug man machallah nala thiate sekhe...,0.00000
10,Wouyyy bb mareme😂😂😂,https://www.youtube.com/watch?v=Myd-Gg4Mhck,wouyyy bb mareme,0.00000
11,Tidiane vrai yamba la 😡 il m’énerve Jtjur,https://www.youtube.com/watch?v=Myd-Gg4Mhck,tidiane vrai yamba la il m’énerve jtjur,0.00000
...,...,...,...,...
396536,Mane cocktail rék bxn😂😂,https://www.youtube.com/watch?v=aj4wylbWqgs,mane cocktail rék bxn,0.25000
396539,Cv nouyou na len ❤,https://www.youtube.com/watch?v=aj4wylbWqgs,cv nouyou na len,0.25000
396540,Fekk lennma fii,https://www.youtube.com/watch?v=aj4wylbWqgs,fekk lennma fii,0.00000
396554,Toppou leine sama guinaw,https://www.youtube.com/watch?v=aj4wylbWqgs,toppou leine sama guinaw,0.25000


In [24]:
clean_data.to_csv(CLEAN_COMMENTS_PATH, index=False)

In [16]:
# import fasttext
# model = fasttext.load_model("lid.176.bin")
# import re
# text = "Mère oumy guène foke ni wérr na"

# tokens = re.findall(r"\w+", text.lower())

# len_tokens = len(tokens)
# fr_count = sum(1 for tok in tokens if model.predict(tok)[0][0] == "__label__fr")
# french_ratio = fr_count / len_tokens if len_tokens > 0 else 0
# print(f"French Ratio: {french_ratio:.2f}")